# Notebook 6 — Perbandingan 3 Metode Deteksi Objek: `SSD MobileNet V2` (FERQUEUE) vs `SSD MobileNet V1` vs `Faster R-CNN ResNet50`

Notebook Google Colab ini membandingkan secara langsung, terukur, dan *end-to-end* **metode utama proyek FERQUEUE** melawan **2 metode deteksi objek pembanding utama** (**MobileNet V1** dan **ResNet50**):

| No | Metode | Backbone CNN | Paradigma Deteksi | Fitur Multi-Skala | Keterangan |
|---:|---|---|---|---|---|
| **1** | **SSD MobileNet V2 FPNLite 320×320** *(Metode Kita — FERQUEUE)* | **MobileNet V2** *(Inverted Residual + Linear Bottleneck)* | Single-Stage | **FPNLite (P3–P7)** | Terlatih khusus domain FERQUEUE (`Mobil`, `Truk`, `Bus`) |
| **2** | **SSD MobileNet V1** *(Pembanding 1 — MobileNet V1)* | **MobileNet V1** *(Depthwise Conv tanpa Skip)* | Single-Stage | SSD Standar (tanpa FPN) | Pretrained COCO resmi TensorFlow (`car→Mobil`, `bus→Bus`, `truck→Truk`) |
| **3** | **Faster R-CNN ResNet50 FPN** *(Pembanding 2 — ResNet50)* | **ResNet50** *(50-layer Residual Network)* | Two-Stage (RPN + RoI) | FPN Standar | Pretrained COCO resmi Torchvision (`car→Mobil`, `bus→Bus`, `truck→Truk`) |

### Apa Saja yang Diuji?
1. **Eksperimen 1 — Benchmark *Live* Backbone CNN (`320×320×3`)**: Mengukur langsung di TensorFlow/Keras jumlah parameter, ukuran bobot FP32 (MB), output channel, dan waktu ekstraksi fitur (`ms`) untuk **MobileNet V2 vs MobileNet V1 vs ResNet50**.
2. **Eksperimen 2 — Evaluasi *Live* Deteksi End-to-End pada Gambar Uji**: Menjalankan ketiga detektor pada gambar uji yang sama dengan ambang deteksi & NMS seragam (`CONF_TH = 0.50`, `NMS_IOU_TH = 0.45`):
   - Jika anotasi COCO `instances_test.json` tersedia → menghitung penuh **mAP@0.50, AP per kelas (`Mobil`, `Truk`, `Bus`), Precision, Recall, F1-Score, & Mean IoU**.
   - Jika hanya folder gambar uji tanpa JSON → otomatis menghitung **Detection Rate (%), Mean Confidence (%), Waktu Inferensi (ms), FPS, Ukuran Bobot (MB), Parameter (Juta), & Skor Komposit (0–100)**.
3. **Output Siap Presentasi (300 DPI + CSV + ZIP)**: 5 grafik resolusi tinggi + tabel matriks keputusan + ekspor ZIP otomatis di Cell terakhir.

### Cara Menjalankan di Google Colab
1. Pilih runtime **GPU (T4)** *(direkomendasikan)* atau CPU melalui `Runtime → Change runtime type`.
2. Klik **`Runtime → Run all`**.
3. Jika model `pcv-model-colab-v4-ckpt-42.zip` atau gambar uji belum ada di Google Drive, Colab otomatis menampilkan tombol *upload*.

## Cell 1 — Setup Library, Cek Hardware (GPU/CPU), & Folder Output

In [ ]:
import json
import math
import shutil
import tarfile
import urllib.request
import zipfile
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import torchvision
from PIL import Image, ImageOps
from torchvision.transforms.functional import to_tensor

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = (
    Path("/content/outputs_presentasi/nb6_mv2_vs_mv1_vs_resnet50")
    if IN_COLAB
    else Path("./outputs_presentasi/nb6_mv2_vs_mv1_vs_resnet50")
)
MODEL_WORK_DIR = (
    Path("/content/ferqueue_nb6_cache")
    if IN_COLAB
    else Path("./outputs_presentasi/_nb6_cache")
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WORK_DIR.mkdir(parents=True, exist_ok=True)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TF_HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0

# Batasi pertumbuhan memori GPU TF agar TF & PyTorch bisa berjalan berdampingan dengan stabil di Colab
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str) -> Path:
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


def ensure_drive_mounted() -> bool:
    if not IN_COLAB:
        return False
    if Path("/content/drive/MyDrive").is_dir():
        return True
    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive").is_dir()


print(
    f"Lingkungan siap! Colab={IN_COLAB} | TF={tf.__version__} (GPU={TF_HAS_GPU}) | "
    f"PyTorch={torch.__version__} (Device={TORCH_DEVICE}) | Torchvision={torchvision.__version__}"
)
print(f"Folder output: {OUT_DIR}")

## Cell 2 — Eksperimen 1 (Benchmark *Live* Backbone): `MobileNet V2` vs `MobileNet V1` vs `ResNet50`

Mengukur langsung ketiga *backbone* ekstraktor fitur di **TensorFlow/Keras** pada ukuran masukan yang sama persis (`320×320×3`):
* **Jumlah Parameter (Juta)** — langsung dari `model.count_params()`
* **Ukuran Bobot FP32 (MB)** — `params × 4 byte`
* **Waktu Ekstraksi Fitur Rata-Rata (`ms`)** — rata-rata 15× *forward pass* setelah 3× *warmup*
* **Kapasitas Representasi Channel (`Output Channels`)** — `1280` pada MobileNet V2 vs `1024` pada MobileNet V1

In [ ]:
BACKBONE_RUNS = 15
dummy_320 = tf.random.uniform((1, 320, 320, 3), minval=-1.0, maxval=1.0, dtype=tf.float32)

backbone_configs = [
    ("MobileNet V2 (FERQUEUE)", "MobileNet V2\n(FERQUEUE ★)", tf.keras.applications.MobileNetV2, "#059669"),
    ("MobileNet V1", "MobileNet V1\n(Pembanding 1)", tf.keras.applications.MobileNet, "#2563EB"),
    ("ResNet50", "ResNet50\n(Pembanding 2)", tf.keras.applications.ResNet50, "#D97706"),
]

backbone_rows = []
for name, short_lbl, builder, color in backbone_configs:
    m = builder(input_shape=(320, 320, 3), include_top=False, weights=None)
    params = int(m.count_params())
    fp32_mb = (params * 4) / (1024 ** 2)
    out_ch = int(m.output_shape[-1])

    for _ in range(3):
        _ = m(dummy_320, training=False).numpy()

    times = []
    for _ in range(BACKBONE_RUNS):
        t0 = perf_counter()
        _ = m(dummy_320, training=False).numpy()
        times.append((perf_counter() - t0) * 1000.0)

    backbone_rows.append({
        "Backbone": name,
        "Label_Plot": short_lbl,
        "Color": color,
        "Parameter_Juta": round(params / 1e6, 2),
        "Ukuran_FP32_MB": round(fp32_mb, 2),
        "Waktu_Backbone_ms": round(float(np.mean(times)), 2),
        "Output_Channel": out_ch,
    })
    del m

backbone_df = pd.DataFrame(backbone_rows)
backbone_df.drop(columns=["Label_Plot", "Color"]).to_csv(OUT_DIR / "00_benchmark_backbone_320.csv", index=False)
display(backbone_df.drop(columns=["Label_Plot", "Color"]))

# Grafik 3 Panel Backbone (300 DPI)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
fig.suptitle("Eksperimen 1: Benchmark Live Backbone CNN (Masukan Seragam 320×320×3)", fontsize=13.5, fontweight="bold", y=1.01)

b_labels = backbone_df["Label_Plot"].tolist()
b_colors = backbone_df["Color"].tolist()

# 1. Parameter
vals = backbone_df["Parameter_Juta"].tolist()
bars = axes[0].bar(b_labels, vals, color=b_colors, width=0.52, edgecolor="#0F172A", linewidth=0.8)
axes[0].set_title("Jumlah Parameter (Juta) ↓ Lebih Kecil Lebih Baik")
axes[0].set_ylabel("Juta Parameter")
axes[0].set_ylim(0, max(vals) * 1.25)
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
for i, (b, v) in enumerate(zip(bars, vals)):
    tag = " ★" if i == int(np.argmin(vals)) else ""
    axes[0].text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03, f"{v:.2f}M{tag}", ha="center", fontweight="bold")

# 2. Ukuran Bobot FP32
vals = backbone_df["Ukuran_FP32_MB"].tolist()
bars = axes[1].bar(b_labels, vals, color=b_colors, width=0.52, edgecolor="#0F172A", linewidth=0.8)
axes[1].set_title("Ukuran Bobot Backbone FP32 (MB) ↓")
axes[1].set_ylabel("Megabyte (MB)")
axes[1].set_ylim(0, max(vals) * 1.25)
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
for i, (b, v) in enumerate(zip(bars, vals)):
    tag = " ★" if i == int(np.argmin(vals)) else ""
    axes[1].text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03, f"{v:.2f} MB{tag}", ha="center", fontweight="bold")

# 3. Waktu Forward Pass
vals = backbone_df["Waktu_Backbone_ms"].tolist()
bars = axes[2].bar(b_labels, vals, color=b_colors, width=0.52, edgecolor="#0F172A", linewidth=0.8)
axes[2].set_title("Waktu Ekstraksi Fitur (ms) ↓ Lebih Cepat Lebih Baik")
axes[2].set_ylabel("Milidetik (ms)")
axes[2].set_ylim(0, max(vals) * 1.28)
axes[2].grid(axis="y", linestyle="--", alpha=0.35)
for i, (b, v) in enumerate(zip(bars, vals)):
    tag = " ★" if i == int(np.argmin(vals)) else ""
    axes[2].text(b.get_x() + b.get_width() / 2, v + max(vals) * 0.03, f"{v:.1f} ms{tag}", ha="center", fontweight="bold")

plt.tight_layout()
save_fig(fig, "00_benchmark_backbone_mv2_mv1_resnet50.png")
plt.show()

## Cell 3 — Muat 3 Model Deteksi Objek End-to-End (`FERQUEUE SSD MobileNet V2` vs `SSD MobileNet V1` vs `Faster R-CNN ResNet50`)

1. **Metode Kita (`SSD MobileNet V2 FPNLite 320×320 — FERQUEUE`)**: dimuat dari `pcv-model-colab-v4-ckpt-42.zip` (Drive / *upload* / lokal).
2. **Pembanding 1 (`SSD MobileNet V1 300×300 — COCO`)**: diunduh otomatis dari arsip resmi TensorFlow Object Detection Model Zoo (`ssd_mobilenet_v1_coco_2018_01_28`), dengan fallback otomatis ke TensorFlow Hub bila diperlukan.
3. **Pembanding 2 (`Faster R-CNN ResNet50 FPN — COCO`)**: dimuat otomatis dari `torchvision.models.detection` bawaan Google Colab.

Kedua pembanding COCO dipetakan secara adil hanya ke 3 kelas kendaraan FERQUEUE (**`car (3) → Mobil`**, **`bus (6) → Bus`**, **`truck (8) → Truk`**) dengan ambang & NMS identik (`CONF_TH = 0.50`, `NMS_IOU_TH = 0.45`).

In [ ]:
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)

MODEL_SOURCE = "auto"  # Pilihan: "auto" | "upload_zip" | "drive" | "local"
DRIVE_MODEL_CANDIDATES = [
    "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/pcv-model-colab-v4-ckpt-42.zip",
    "/content/drive/MyDrive/ferqueue/pcv-model-colab-v4-ckpt-42.zip",
    "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-35.zip",
    "/content/drive/MyDrive/ferqueue/pcv-model-colab-v3-ckpt-35.zip",
]
LOCAL_MODEL_CANDIDATES = [
    "models/mobilenet/runs/pcv-receiver-v4/exports/imported-ckpt-42",
    "outputs/pcv-model-colab-v4-ckpt-42.zip",
    "models/mobilenet/runs/pcv-receiver-v3-unfrozen/exports/imported-ckpt-35",
    "outputs/pcv-model-colab-v3-ckpt-35.zip",
]

CLASSES = ["Mobil", "Truk", "Bus"]
CLASS_COLORS = {
    "Mobil": (34, 197, 94),   # Hijau
    "Truk": (245, 158, 11),   # Oranye
    "Bus": (59, 130, 246),    # Biru
}
CONF_TH = 0.50
NMS_IOU_TH = 0.45
MAX_DET = 10
COCO_VEHICLE_MAP = {
    3: "Mobil",  # car
    6: "Bus",    # bus
    8: "Truk",   # truck
}


# ------------------------------------------------------------------
# 1. MUAT METODE KITA: FERQUEUE SSD MobileNet V2 FPNLite 320x320
# ------------------------------------------------------------------
def resolve_ferqueue_model(src_mode: str):
    extract_dir = MODEL_WORK_DIR / "ferqueue_extracted"
    meta = {
        "classes": CLASSES,
        "model": "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8",
        "checkpoint": "ckpt-42",
        "preprocessing": {"denoise_kernel": 0, "clahe_clip_limit": 0, "clahe_grid_size": 8, "gamma": 1.0},
    }

    cached_saved = extract_dir / "saved_model"
    if src_mode == "auto" and (cached_saved / "saved_model.pb").is_file():
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        print(f"[1/3] Memakai cache SavedModel FERQUEUE: {cached_saved}")
        return cached_saved, meta

    target_path = None
    if src_mode == "upload_zip" and IN_COLAB:
        print("Silakan unggah ZIP model FERQUEUE (mis. pcv-model-colab-v4-ckpt-42.zip)...")
        uploaded = colab_files.upload()
        zip_name, content = next(iter(uploaded.items()))
        target_path = MODEL_WORK_DIR / Path(zip_name).name
        target_path.write_bytes(content)
    elif src_mode in ("drive", "auto") and IN_COLAB:
        if ensure_drive_mounted():
            for cand in DRIVE_MODEL_CANDIDATES:
                p = Path(cand)
                if p.exists():
                    target_path = p
                    print(f"[1/3] Model FERQUEUE ditemukan di Google Drive: {target_path}")
                    break
        if target_path is None:
            if src_mode == "drive":
                raise FileNotFoundError(f"Model tidak ditemukan di Google Drive: {DRIVE_MODEL_CANDIDATES[0]}")
            print("Model belum ditemukan di Drive. Silakan unggah ZIP model FERQUEUE (mis. pcv-model-colab-v4-ckpt-42.zip)...")
            uploaded = colab_files.upload()
            zip_name, content = next(iter(uploaded.items()))
            target_path = MODEL_WORK_DIR / Path(zip_name).name
            target_path.write_bytes(content)
    else:
        for cand in LOCAL_MODEL_CANDIDATES:
            p = Path(cand)
            if p.exists():
                target_path = p
                break
        if target_path is None:
            raise FileNotFoundError("Model lokal FERQUEUE tidak ditemukan. Sesuaikan LOCAL_MODEL_CANDIDATES.")

    if target_path.is_file() and target_path.suffix.lower() == ".zip":
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_path) as zf:
            zf.extractall(extract_dir)
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        saved_dir = extract_dir / "saved_model"
    elif target_path.is_dir():
        saved_dir = target_path / "saved_model" if (target_path / "saved_model/saved_model.pb").is_file() else target_path
        for meta_file in (target_path / "model.json", target_path.parent.parent / "export.json"):
            if meta_file.is_file():
                meta.update(json.loads(meta_file.read_text(encoding="utf-8")))
                break
    else:
        raise FileNotFoundError(f"Path model tidak valid: {target_path}")

    if not (saved_dir / "saved_model.pb").is_file():
        raise FileNotFoundError(f"saved_model.pb tidak ditemukan di {saved_dir}")
    return saved_dir, meta


t0 = perf_counter()
ferqueue_saved_dir, ferqueue_meta = resolve_ferqueue_model(MODEL_SOURCE)
CLASSES = list(ferqueue_meta.get("classes", CLASSES))
MODEL_PREPROCESSING = dict(ferqueue_meta.get("preprocessing", {}))
ferqueue_tf_model = tf.saved_model.load(str(ferqueue_saved_dir))
ferqueue_detect_fn = ferqueue_tf_model.signatures["serving_default"]
ferqueue_params = sum(int(np.prod(v.shape)) for v in ferqueue_detect_fn.variables)
ferqueue_fp32_mb = (ferqueue_params * 4) / (1024 ** 2)
print(
    f"[1/3] FERQUEUE SSD MobileNet V2 FPNLite siap ({perf_counter() - t0:.2f} dtk) | "
    f"{ferqueue_params / 1e6:.2f}M params | {ferqueue_fp32_mb:.2f} MB FP32"
)


# ------------------------------------------------------------------
# 2. MUAT PEMBANDING 1: SSD MobileNet V1 (Resmi TensorFlow Model Zoo)
# ------------------------------------------------------------------
SSD_MV1_URL = "http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v1_coco_2018_01_28.tar.gz"
SSD_MV1_DIR = MODEL_WORK_DIR / "ssd_mobilenet_v1_coco_2018_01_28"


def load_ssd_mobilenet_v1():
    saved_dir = SSD_MV1_DIR / "saved_model"
    if not (saved_dir / "saved_model.pb").is_file():
        tar_path = MODEL_WORK_DIR / "ssd_mobilenet_v1_coco_2018_01_28.tar.gz"
        if not tar_path.is_file():
            print("[2/3] Mengunduh SSD MobileNet V1 (COCO) resmi TensorFlow...")
            urllib.request.urlretrieve(SSD_MV1_URL, tar_path)
        with tarfile.open(tar_path, "r:gz") as tf_tar:
            tf_tar.extractall(MODEL_WORK_DIR)

    pb_size_mb = (saved_dir / "saved_model.pb").stat().st_size / (1024 ** 2)
    loaded = tf.saved_model.load(str(saved_dir))
    fn = loaded.signatures["serving_default"]
    input_keys = list(fn.structured_input_signature[1].keys())
    input_key = input_keys[0] if input_keys else None
    n_vars = sum(int(np.prod(v.shape)) for v in fn.variables)
    # TF1 SavedModel membekukan bobot ke node Const di saved_model.pb (~6.81M params, ~27.8 MB)
    est_params = n_vars if n_vars > 0 else int((pb_size_mb * (1024 ** 2)) / 4)
    return loaded, fn, input_key, est_params, pb_size_mb


t1 = perf_counter()
try:
    mv1_tf_model, mv1_detect_fn, mv1_input_key, mv1_params, mv1_size_mb = load_ssd_mobilenet_v1()
except Exception as err:
    print(f"Unduhan TF Model Zoo gagal ({err}), beralih ke TensorFlow Hub SSD MobileNet V1...")
    import tensorflow_hub as hub
    mv1_tf_model = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v1/fpn_640x640/1")
    mv1_detect_fn = mv1_tf_model.signatures["serving_default"]
    input_keys = list(mv1_detect_fn.structured_input_signature[1].keys())
    mv1_input_key = input_keys[0] if input_keys else None
    mv1_params = sum(int(np.prod(v.shape)) for v in mv1_detect_fn.variables) or 6_810_000
    mv1_size_mb = (mv1_params * 4) / (1024 ** 2)

print(
    f"[2/3] SSD MobileNet V1 siap ({perf_counter() - t1:.2f} dtk) | "
    f"{mv1_params / 1e6:.2f}M params | {mv1_size_mb:.2f} MB"
)


# ------------------------------------------------------------------
# 3. MUAT PEMBANDING 2: Faster R-CNN ResNet50 FPN (Resmi Torchvision)
# ------------------------------------------------------------------
t2 = perf_counter()
resnet_model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT
).to(TORCH_DEVICE).eval()
resnet_params = sum(p.numel() for p in resnet_model.parameters())
resnet_fp32_mb = (resnet_params * 4) / (1024 ** 2)
print(
    f"[3/3] Faster R-CNN ResNet50 FPN siap ({perf_counter() - t2:.2f} dtk) | "
    f"{resnet_params / 1e6:.2f}M params | {resnet_fp32_mb:.2f} MB FP32"
)


# ------------------------------------------------------------------
# KONFIGURASI METODE & FUNGSI INFERENSI TERSTANDAR
# ------------------------------------------------------------------
METHOD_NAMES = [
    "SSD MobileNet V2 FPNLite (FERQUEUE)",
    "SSD MobileNet V1",
    "Faster R-CNN ResNet50 FPN",
]
METHOD_SHORT = {
    METHOD_NAMES[0]: "FERQUEUE\nSSD MobileNet V2",
    METHOD_NAMES[1]: "Pembanding 1\nSSD MobileNet V1",
    METHOD_NAMES[2]: "Pembanding 2\nResNet50 FPN",
}
METHOD_COLORS = {
    METHOD_NAMES[0]: "#059669",  # Hijau Emerald (Metode Kita)
    METHOD_NAMES[1]: "#2563EB",  # Biru
    METHOD_NAMES[2]: "#D97706",  # Oranye Amber
}
MODEL_SPECS = {
    METHOD_NAMES[0]: {
        "backbone": "MobileNet V2",
        "tipe": "Single-Stage + FPNLite",
        "params_m": round(ferqueue_params / 1e6, 2),
        "size_mb": round(ferqueue_fp32_mb, 2),
    },
    METHOD_NAMES[1]: {
        "backbone": "MobileNet V1",
        "tipe": "Single-Stage (Tanpa FPN)",
        "params_m": round(mv1_params / 1e6, 2),
        "size_mb": round(mv1_size_mb, 2),
    },
    METHOD_NAMES[2]: {
        "backbone": "ResNet50",
        "tipe": "Two-Stage (RPN + FPN)",
        "params_m": round(resnet_params / 1e6, 2),
        "size_mb": round(resnet_fp32_mb, 2),
    },
}


def compute_iou_xyxy(box_a, box_b) -> float:
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter <= 0.0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return float(inter / max(1e-9, area_a + area_b - inter))


def apply_nms(dets: list[dict], iou_th: float = NMS_IOU_TH, max_det: int = MAX_DET) -> list[dict]:
    ordered = sorted(dets, key=lambda d: d["score"], reverse=True)
    kept = []
    for cand in ordered:
        if all(compute_iou_xyxy(cand["xyxy_px"], prev["xyxy_px"]) < iou_th for prev in kept):
            kept.append(cand)
        if len(kept) >= max_det:
            break
    return kept


def preprocess_ferqueue_rgb(rgb: np.ndarray) -> np.ndarray:
    img = rgb.copy()
    k = int(MODEL_PREPROCESSING.get("denoise_kernel", 0) or 0)
    if k >= 3 and k % 2 == 1:
        img = cv2.GaussianBlur(img, (k, k), 0)
    clip = float(MODEL_PREPROCESSING.get("clahe_clip_limit", 0) or 0)
    if clip > 0:
        grid = int(MODEL_PREPROCESSING.get("clahe_grid_size", 8) or 8)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l_ch, a_ch, b_ch = cv2.split(lab)
        l_ch = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l_ch)
        img = cv2.cvtColor(cv2.merge((l_ch, a_ch, b_ch)), cv2.COLOR_LAB2RGB)
    gamma = float(MODEL_PREPROCESSING.get("gamma", 1.0) or 1.0)
    if abs(gamma - 1.0) > 1e-3:
        lut = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in range(256)], dtype=np.uint8)
        img = cv2.LUT(img, lut)
    return img


def predict_ferqueue_mv2(rgb: np.ndarray, conf_th: float = CONF_TH):
    h, w = rgb.shape[:2]
    proc = preprocess_ferqueue_rgb(rgb)
    tensor = tf.convert_to_tensor(proc[None, ...], dtype=tf.uint8)
    t_start = perf_counter()
    outputs = ferqueue_detect_fn(tensor)
    boxes = outputs["detection_boxes"].numpy()[0]
    classes = outputs["detection_classes"].numpy()[0].astype(int)
    scores = outputs["detection_scores"].numpy()[0]
    elapsed_ms = (perf_counter() - t_start) * 1000.0

    dets = []
    for box, cls_id, score in zip(boxes, classes, scores):
        score_f = float(score)
        if score_f < conf_th:
            continue
        if 1 <= cls_id <= len(CLASSES):
            y1, x1, y2, x2 = [float(v) for v in box]
            dets.append({
                "class": CLASSES[cls_id - 1],
                "score": score_f,
                "xyxy_px": [x1 * w, y1 * h, x2 * w, y2 * h],
            })
    return apply_nms(dets), elapsed_ms


def predict_ssd_mv1(rgb: np.ndarray, conf_th: float = CONF_TH):
    h, w = rgb.shape[:2]
    tensor = tf.convert_to_tensor(rgb[None, ...], dtype=tf.uint8)
    t_start = perf_counter()
    outputs = mv1_detect_fn(**{mv1_input_key: tensor}) if mv1_input_key else mv1_detect_fn(tensor)
    boxes = outputs["detection_boxes"].numpy()[0]
    classes = outputs["detection_classes"].numpy()[0].astype(int)
    scores = outputs["detection_scores"].numpy()[0]
    elapsed_ms = (perf_counter() - t_start) * 1000.0

    dets = []
    for box, cls_id, score in zip(boxes, classes, scores):
        score_f = float(score)
        cls_name = COCO_VEHICLE_MAP.get(int(cls_id))
        if cls_name is None or score_f < conf_th:
            continue
        y1, x1, y2, x2 = [float(v) for v in box]
        dets.append({
            "class": cls_name,
            "score": score_f,
            "xyxy_px": [x1 * w, y1 * h, x2 * w, y2 * h],
        })
    return apply_nms(dets), elapsed_ms


@torch.inference_mode()
def predict_faster_rcnn_resnet50(rgb: np.ndarray, conf_th: float = CONF_TH):
    inp = to_tensor(rgb).to(TORCH_DEVICE)
    if TORCH_DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_start = perf_counter()
    out = resnet_model([inp])[0]
    if TORCH_DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed_ms = (perf_counter() - t_start) * 1000.0

    boxes = out["boxes"].detach().cpu().numpy()
    labels = out["labels"].detach().cpu().numpy()
    scores = out["scores"].detach().cpu().numpy()

    dets = []
    for box, cls_id, score in zip(boxes, labels, scores):
        score_f = float(score)
        cls_name = COCO_VEHICLE_MAP.get(int(cls_id))
        if cls_name is None or score_f < conf_th:
            continue
        x1, y1, x2, y2 = [float(v) for v in box]
        dets.append({
            "class": cls_name,
            "score": score_f,
            "xyxy_px": [x1, y1, x2, y2],
        })
    return apply_nms(dets), elapsed_ms


PREDICTORS = {
    METHOD_NAMES[0]: lambda img: predict_ferqueue_mv2(img),
    METHOD_NAMES[1]: lambda img: predict_ssd_mv1(img),
    METHOD_NAMES[2]: lambda img: predict_faster_rcnn_resnet50(img),
}

spec_df = pd.DataFrame([
    {
        "Metode": m,
        "Backbone": v["backbone"],
        "Tipe Arsitektur": v["tipe"],
        "Parameter Full (Juta)": v["params_m"],
        "Ukuran Model (MB)": v["size_mb"],
    }
    for m, v in MODEL_SPECS.items()
])
display(spec_df)

## Cell 4 — Muat Gambar Uji & (Opsional) Anotasi Ground Truth COCO

* `IMAGE_SOURCE = "auto"`:
  - Di **Google Colab**, otomatis mencari folder gambar uji di Google Drive (`MyDrive/ferqueue/gambar`, `MyDrive/ferqueue/test`, `MyDrive/ferqueue/helper`). Jika belum ada di Drive, otomatis memunculkan dialog *upload* gambar atau file `.zip`.
  - Di **lokal**, otomatis memakai gambar uji dari repo (`data/raw/vehicles` sesuai `instances_test.json` atau `data/processed/pcv-notebook-samples`).
* `GT_COCO_PATH = "auto"`: jika file `instances_test.json` ditemukan (di Drive atau lokal), otomatis menghitung **mAP@0.50, AP per kelas (`Mobil`, `Truk`, `Bus`), Precision, Recall, F1-Score, & Mean IoU**. Jika tanpa JSON anotasi, otomatis berjalan pada mode evaluasi lapangan.
* `MAX_TEST_IMAGES = 30`: batas sampel agar eksekusi cepat & seimbang antar kelas (ubah ke `None` untuk semua gambar).

In [ ]:
IMAGE_SOURCE = "auto"     # Pilihan: "auto" | "drive" | "upload" | "local"
GT_COCO_PATH = "auto"     # Pilihan: "auto" | path file instances_test.json | None
MAX_TEST_IMAGES = 30      # Ubah ke None untuk menguji seluruh folder

DRIVE_IMAGE_DIRS = [
    "/content/drive/MyDrive/ferqueue/gambar",
    "/content/drive/MyDrive/ferqueue/test",
    "/content/drive/MyDrive/ferqueue/vehicles",
    "/content/drive/MyDrive/ferqueue/helper",
]
DRIVE_GT_CANDIDATES = [
    "/content/drive/MyDrive/ferqueue/instances_test.json",
    "/content/drive/MyDrive/ferqueue/pcv-coco-v4/annotations/instances_test.json",
    "/content/drive/MyDrive/ferqueue/pcv-coco-v3/annotations/instances_test.json",
]
LOCAL_IMAGE_DIRS = [
    "data/raw/vehicles",
    "data/processed/pcv-notebook-samples",
    "data/helper",
]
LOCAL_GT_CANDIDATES = [
    "data/processed/pcv-coco-v4/annotations/instances_test.json",
    "data/processed/pcv-coco-v3/annotations/instances_test.json",
    "data/processed/pcv-coco-v2/annotations/instances_test.json",
]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def load_rgb_image(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        return np.asarray(im, dtype=np.uint8)


def find_gt_json():
    if GT_COCO_PATH is None:
        return None
    if GT_COCO_PATH != "auto":
        p = Path(GT_COCO_PATH)
        return p if p.is_file() else None
    cands = DRIVE_GT_CANDIDATES if IN_COLAB else LOCAL_GT_CANDIDATES
    if IN_COLAB:
        ensure_drive_mounted()
    for c in cands:
        p = Path(c)
        if p.is_file():
            return p
    return None


def parse_coco_gt(gt_path: Path) -> dict[str, list[dict]]:
    data = json.loads(gt_path.read_text(encoding="utf-8"))
    cat_map = {int(c["id"]): str(c["name"]) for c in data.get("categories", [])}
    id_to_name = {int(im["id"]): Path(im["file_name"]).name for im in data.get("images", [])}
    gt_by_name = {name: [] for name in id_to_name.values()}
    for ann in data.get("annotations", []):
        name = id_to_name.get(int(ann["image_id"]))
        cls_name = cat_map.get(int(ann["category_id"]))
        if name is None or cls_name not in CLASSES:
            continue
        x, y, bw, bh = [float(v) for v in ann["bbox"]]
        gt_by_name[name].append({"class": cls_name, "xyxy_px": [x, y, x + bw, y + bh]})
    return gt_by_name


def collect_image_paths(src_mode: str) -> list[Path]:
    upload_dir = MODEL_WORK_DIR / "uploaded_images"
    if src_mode == "upload" and IN_COLAB:
        upload_dir.mkdir(parents=True, exist_ok=True)
        print("Silakan unggah gambar uji (.jpg/.png) atau 1 file .zip berisi gambar uji...")
        uploaded = colab_files.upload()
        for fname, content in uploaded.items():
            dst = upload_dir / Path(fname).name
            dst.write_bytes(content)
            if dst.suffix.lower() == ".zip":
                with zipfile.ZipFile(dst) as zf:
                    zf.extractall(upload_dir)
        return sorted(p for p in upload_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

    if src_mode in ("drive", "auto") and IN_COLAB:
        if ensure_drive_mounted():
            for d in DRIVE_IMAGE_DIRS:
                p = Path(d)
                if p.is_dir():
                    imgs = sorted(x for x in p.rglob("*") if x.is_file() and x.suffix.lower() in IMAGE_EXTS)
                    if imgs:
                        print(f"Folder gambar uji ditemukan di Drive: {p} ({len(imgs)} file)")
                        return imgs
        if src_mode == "drive":
            raise FileNotFoundError("Folder gambar uji tidak ditemukan di Google Drive.")
        return collect_image_paths("upload")

    for d in LOCAL_IMAGE_DIRS:
        p = Path(d)
        if p.is_dir():
            imgs = sorted(x for x in p.rglob("*") if x.is_file() and x.suffix.lower() in IMAGE_EXTS)
            if imgs:
                print(f"Folder gambar uji lokal ditemukan: {p} ({len(imgs)} file)")
                return imgs
    raise FileNotFoundError("Tidak ada gambar uji ditemukan.")


all_img_paths = collect_image_paths(IMAGE_SOURCE)
gt_json_path = find_gt_json()
gt_by_filename = parse_coco_gt(gt_json_path) if gt_json_path else {}

matched_paths = [p for p in all_img_paths if p.name in gt_by_filename and len(gt_by_filename[p.name]) > 0]
HAS_GROUND_TRUTH = len(matched_paths) > 0

if HAS_GROUND_TRUTH:
    if MAX_TEST_IMAGES is not None and len(matched_paths) > MAX_TEST_IMAGES:
        per_cls = {c: [] for c in CLASSES}
        for p in matched_paths:
            main_cls = gt_by_filename[p.name][0]["class"]
            per_cls.setdefault(main_cls, []).append(p)
        selected = []
        quota = max(1, math.ceil(MAX_TEST_IMAGES / len(CLASSES)))
        for c in CLASSES:
            selected.extend(per_cls.get(c, [])[:quota])
        selected_paths = selected[:MAX_TEST_IMAGES]
    else:
        selected_paths = matched_paths
    print(f"Mode Evaluasi: FULL GROUND-TRUTH COCO ({gt_json_path.name}) | {len(selected_paths)} gambar uji terseleksi")
else:
    if MAX_TEST_IMAGES is not None and len(all_img_paths) > MAX_TEST_IMAGES:
        step = max(1, len(all_img_paths) // MAX_TEST_IMAGES)
        selected_paths = all_img_paths[::step][:MAX_TEST_IMAGES]
    else:
        selected_paths = all_img_paths
    print(f"Mode Evaluasi: LAPANGAN / TANPA JSON ANOTASI | {len(selected_paths)} gambar uji terseleksi")

test_samples = []
for p in selected_paths:
    rgb = load_rgb_image(p)
    test_samples.append({
        "name": p.name,
        "path": p,
        "rgb": rgb,
        "gt": gt_by_filename.get(p.name, []) if HAS_GROUND_TRUTH else None,
    })

print(
    f"Total gambar siap diuji: {len(test_samples)} | "
    f"Contoh resolusi: {test_samples[0]['rgb'].shape[1]}×{test_samples[0]['rgb'].shape[0]}"
)

## Cell 5 — Eksperimen 2: Benchmark *Live* Deteksi End-to-End & Penentuan Peringkat

Menjalankan seluruh gambar uji pada ketiga detektor (`SSD MobileNet V2 FPNLite FERQUEUE` vs `SSD MobileNet V1` vs `Faster R-CNN ResNet50 FPN`) setelah *warmup* `3×`, lalu menghitung:
* **Kualitas Deteksi**: `mAP@0.50`, `AP per kelas`, `Precision`, `Recall`, `F1-Score`, `Mean IoU` *(bila ada GT COCO)* atau `Detection Rate (%)` & `Mean Confidence (%)` *(bila tanpa GT)*
* **Kecepatan**: `Waktu Inferensi (ms/gambar)` & `Throughput (FPS)`
* **Efisiensi Model**: `Ukuran Bobot (MB)` & `Jumlah Parameter (Juta)`
* **Skor Komposit Keseluruhan (0–100)**: **50% Kualitas Deteksi + 25% Kecepatan (FPS) + 25% Efisiensi Bobot & Parameter**

In [ ]:
WARMUP_RUNS = 3
warmup_img = test_samples[0]["rgb"]
print(f"Melakukan warmup ({WARMUP_RUNS}×) pada 3 model deteksi...")
for m_name in METHOD_NAMES:
    for _ in range(WARMUP_RUNS):
        PREDICTORS[m_name](warmup_img)

predictions = {m: [] for m in METHOD_NAMES}
latencies_ms = {m: [] for m in METHOD_NAMES}
detail_rows = []

for idx, sample in enumerate(test_samples):
    rgb = sample["rgb"]
    for m_name in METHOD_NAMES:
        dets, ms = PREDICTORS[m_name](rgb)
        predictions[m_name].append(dets)
        latencies_ms[m_name].append(ms)
        top_det = dets[0] if dets else None
        detail_rows.append({
            "No": idx + 1,
            "Gambar": sample["name"],
            "Metode": m_name,
            "Jumlah_Deteksi": len(dets),
            "Prediksi_Teratas": top_det["class"] if top_det else "Tidak Terdeteksi",
            "Confidence_Teratas": round(top_det["score"], 4) if top_det else 0.0,
            "Waktu_ms": round(ms, 2),
        })

print("Inferensi seluruh gambar selesai!")


def evaluate_against_gt(method_preds: list[list[dict]], samples: list[dict], iou_th: float = 0.50) -> dict:
    per_class_ap = {}
    total_tp = total_fp = total_fn = 0
    matched_ious = []

    for cls_name in CLASSES:
        gt_records = {}
        n_gt_cls = 0
        for img_i, sample in enumerate(samples):
            boxes = [g["xyxy_px"] for g in (sample["gt"] or []) if g["class"] == cls_name]
            gt_records[img_i] = {"boxes": boxes, "matched": [False] * len(boxes)}
            n_gt_cls += len(boxes)

        cls_preds = []
        for img_i, dets in enumerate(method_preds):
            for d in dets:
                if d["class"] == cls_name:
                    cls_preds.append((img_i, d["score"], d["xyxy_px"]))
        cls_preds.sort(key=lambda x: x[1], reverse=True)

        tp_arr = np.zeros(len(cls_preds), dtype=np.float32)
        fp_arr = np.zeros(len(cls_preds), dtype=np.float32)

        for p_i, (img_i, _score, p_box) in enumerate(cls_preds):
            rec = gt_records[img_i]
            best_iou = 0.0
            best_gt_idx = -1
            for gt_idx, gt_box in enumerate(rec["boxes"]):
                if rec["matched"][gt_idx]:
                    continue
                iou = compute_iou_xyxy(p_box, gt_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx
            if best_iou >= iou_th and best_gt_idx >= 0:
                rec["matched"][best_gt_idx] = True
                tp_arr[p_i] = 1.0
                matched_ious.append(best_iou)
            else:
                fp_arr[p_i] = 1.0

        tp_cls = int(tp_arr.sum())
        fp_cls = int(fp_arr.sum())
        fn_cls = max(0, n_gt_cls - tp_cls)
        total_tp += tp_cls
        total_fp += fp_cls
        total_fn += fn_cls

        if n_gt_cls == 0 or len(cls_preds) == 0:
            per_class_ap[cls_name] = 0.0
            continue

        cum_tp = np.cumsum(tp_arr)
        cum_fp = np.cumsum(fp_arr)
        recalls = cum_tp / max(1, n_gt_cls)
        precisions = cum_tp / np.maximum(1e-9, cum_tp + cum_fp)

        recall_levels = np.linspace(0.0, 1.0, 101)
        ap = float(np.mean([precisions[recalls >= r].max() if np.any(recalls >= r) else 0.0 for r in recall_levels]))
        per_class_ap[cls_name] = ap

    precision = total_tp / max(1, total_tp + total_fp)
    recall = total_tp / max(1, total_tp + total_fn)
    f1 = (2 * precision * recall) / max(1e-9, precision + recall)
    valid_aps = [v for k, v in per_class_ap.items() if any(g["class"] == k for s in samples for g in (s["gt"] or []))]
    map50 = float(np.mean(valid_aps)) if valid_aps else 0.0
    mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0
    return {
        "mAP50_pct": round(map50 * 100.0, 2),
        "Precision_pct": round(precision * 100.0, 2),
        "Recall_pct": round(recall * 100.0, 2),
        "F1_pct": round(f1 * 100.0, 2),
        "Mean_IoU_pct": round(mean_iou * 100.0, 2),
        "AP_Mobil_pct": round(per_class_ap.get("Mobil", 0.0) * 100.0, 2),
        "AP_Truk_pct": round(per_class_ap.get("Truk", 0.0) * 100.0, 2),
        "AP_Bus_pct": round(per_class_ap.get("Bus", 0.0) * 100.0, 2),
    }


summary_rows = []
for m_name in METHOD_NAMES:
    m_preds = predictions[m_name]
    m_lat = np.asarray(latencies_ms[m_name], dtype=np.float64)
    avg_ms = float(np.mean(m_lat))
    fps = 1000.0 / max(1e-6, avg_ms)

    detected_imgs = sum(1 for d in m_preds if len(d) > 0)
    det_rate = (detected_imgs / max(1, len(test_samples))) * 100.0
    all_scores = [d["score"] for dets in m_preds for d in dets]
    mean_conf = (float(np.mean(all_scores)) * 100.0) if all_scores else 0.0

    row = {
        "Metode": m_name,
        "Backbone": MODEL_SPECS[m_name]["backbone"],
        "Tipe": MODEL_SPECS[m_name]["tipe"],
        "Parameter_Juta": MODEL_SPECS[m_name]["params_m"],
        "Ukuran_MB": MODEL_SPECS[m_name]["size_mb"],
        "Waktu_ms": round(avg_ms, 2),
        "FPS": round(fps, 2),
        "Detection_Rate_pct": round(det_rate, 2),
        "Mean_Confidence_pct": round(mean_conf, 2),
    }
    if HAS_GROUND_TRUTH:
        row.update(evaluate_against_gt(m_preds, test_samples, iou_th=0.50))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

max_fps = max(float(summary_df["FPS"].max()), 1e-6)
min_size = min(float(summary_df["Ukuran_MB"].min()), 1e9)
min_params = min(float(summary_df["Parameter_Juta"].min()), 1e9)

composite_scores = []
for _, r in summary_df.iterrows():
    if HAS_GROUND_TRUTH:
        quality_score = 0.65 * float(r["mAP50_pct"]) + 0.35 * float(r["F1_pct"])
    else:
        quality_score = 0.65 * float(r["Detection_Rate_pct"]) + 0.35 * float(r["Mean_Confidence_pct"])
    speed_score = (float(r["FPS"]) / max_fps) * 100.0
    size_eff = (min_size / max(1e-6, float(r["Ukuran_MB"]))) * 100.0
    param_eff = (min_params / max(1e-6, float(r["Parameter_Juta"]))) * 100.0
    efficiency_score = 0.5 * size_eff + 0.5 * param_eff
    overall = 0.50 * quality_score + 0.25 * speed_score + 0.25 * efficiency_score
    composite_scores.append(round(overall, 2))

summary_df["Skor_Komposit"] = composite_scores
summary_df = summary_df.sort_values("Skor_Komposit", ascending=False).reset_index(drop=True)
summary_df.insert(0, "Peringkat", [f"#{i + 1}" for i in range(len(summary_df))])

summary_df.to_csv(OUT_DIR / "01_ringkasan_mv2_vs_mv1_vs_resnet50.csv", index=False)
pd.DataFrame(detail_rows).to_csv(OUT_DIR / "02_detail_deteksi_per_gambar.csv", index=False)

winner = summary_df.iloc[0]
print(f"\n=== HASIL AKHIR ({len(test_samples)} GAMBAR UJI) ===")
print(f"PEMENANG PERINGKAT #1 : {winner['Metode']} (Skor Komposit: {winner['Skor_Komposit']:.2f}/100)")
display(summary_df)

## Cell 6 — Grafik Perbandingan Kuantitatif 4 Panel (300 DPI)

Membandingkan secara langsung `SSD MobileNet V2 FPNLite (FERQUEUE)` vs `SSD MobileNet V1` vs `Faster R-CNN ResNet50 FPN` pada 4 metrik utama: Akurasi/Deteksi, F1/Confidence, Kecepatan FPS & Latency, serta Ukuran Bobot Model & Jumlah Parameter.

In [ ]:
ordered_df = summary_df.set_index("Metode").loc[METHOD_NAMES].reset_index()
x_labels = [METHOD_SHORT[m] for m in METHOD_NAMES]
colors = [METHOD_COLORS[m] for m in METHOD_NAMES]

fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.2))
fig.suptitle(
    "Perbandingan Kuantitatif: SSD MobileNet V2 (FERQUEUE) vs SSD MobileNet V1 vs Faster R-CNN ResNet50",
    fontsize=13.5,
    fontweight="bold",
    y=0.98,
)

# Panel 1: Akurasi Utama
ax = axes[0, 0]
col1 = "mAP50_pct" if HAS_GROUND_TRUTH else "Detection_Rate_pct"
title1 = "1. Akurasi Deteksi — mAP@0.50 (%) ↑" if HAS_GROUND_TRUTH else "1. Keberhasilan Deteksi — Detection Rate (%) ↑"
vals1 = ordered_df[col1].tolist()
bars1 = ax.bar(x_labels, vals1, color=colors, width=0.55, edgecolor="#0F172A", linewidth=0.8)
ax.set_title(title1)
ax.set_ylim(0, max(108, max(vals1) * 1.18))
ax.set_ylabel("Persentase (%)")
ax.grid(axis="y", linestyle="--", alpha=0.35)
best1 = int(np.argmax(vals1))
for i, (b, v) in enumerate(zip(bars1, vals1)):
    tag = "  ★" if i == best1 else ""
    ax.text(b.get_x() + b.get_width() / 2, v + 2.2, f"{v:.1f}%{tag}", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 2: F1-Score / Mean Confidence
ax = axes[0, 1]
col2 = "F1_pct" if HAS_GROUND_TRUTH else "Mean_Confidence_pct"
title2 = "2. Keseimbangan Deteksi — F1-Score (%) ↑" if HAS_GROUND_TRUTH else "2. Keyakinan Prediksi — Mean Confidence (%) ↑"
vals2 = ordered_df[col2].tolist()
bars2 = ax.bar(x_labels, vals2, color=colors, width=0.55, edgecolor="#0F172A", linewidth=0.8)
ax.set_title(title2)
ax.set_ylim(0, max(108, max(vals2) * 1.18))
ax.set_ylabel("Persentase (%)")
ax.grid(axis="y", linestyle="--", alpha=0.35)
best2 = int(np.argmax(vals2))
for i, (b, v) in enumerate(zip(bars2, vals2)):
    tag = "  ★" if i == best2 else ""
    ax.text(b.get_x() + b.get_width() / 2, v + 2.2, f"{v:.1f}%{tag}", ha="center", va="bottom", fontweight="bold", fontsize=10)

# Panel 3: Kecepatan Inferensi (FPS & ms)
ax = axes[1, 0]
vals3 = ordered_df["FPS"].tolist()
ms_list = ordered_df["Waktu_ms"].tolist()
bars3 = ax.bar(x_labels, vals3, color=colors, width=0.55, edgecolor="#0F172A", linewidth=0.8)
ax.set_title(f"3. Kecepatan Inferensi — Throughput FPS ({TORCH_DEVICE.type.upper()}) ↑")
ax.set_ylim(0, max(vals3) * 1.28 + 1)
ax.set_ylabel("Frame per Detik (FPS)")
ax.grid(axis="y", linestyle="--", alpha=0.35)
best3 = int(np.argmax(vals3))
for i, (b, v, ms) in enumerate(zip(bars3, vals3, ms_list)):
    tag = " ★" if i == best3 else ""
    ax.text(
        b.get_x() + b.get_width() / 2,
        v + max(vals3) * 0.03,
        f"{v:.1f} FPS{tag}\n({ms:.1f} ms)",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=9.5,
    )

# Panel 4: Ukuran Bobot (MB) & Parameter (Juta)
ax = axes[1, 1]
vals4 = ordered_df["Ukuran_MB"].tolist()
params_list = ordered_df["Parameter_Juta"].tolist()
bars4 = ax.bar(x_labels, vals4, color=colors, width=0.55, edgecolor="#0F172A", linewidth=0.8)
ax.set_title("4. Efisiensi Model — Ukuran Bobot (MB) ↓ Lebih Kecil Lebih Baik")
ax.set_ylim(0, max(vals4) * 1.28 + 5)
ax.set_ylabel("Ukuran Bobot (MB)")
ax.grid(axis="y", linestyle="--", alpha=0.35)
best4 = int(np.argmin(vals4))
for i, (b, v, pm) in enumerate(zip(bars4, vals4, params_list)):
    tag = " ★ TERINGAN" if i == best4 else ""
    ax.text(
        b.get_x() + b.get_width() / 2,
        v + max(vals4) * 0.03,
        f"{v:.1f} MB{tag}\n({pm:.2f}M params)",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=9.5,
    )

plt.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, "01_perbandingan_kuantitatif_4panel.png")
plt.show()

## Cell 7 — Radar Chart 5 Dimensi & Peringkat Skor Komposit (300 DPI)

Menampilkan profil keseluruhan 5 kriteria (**Akurasi/Deteksi**, **F1/Confidence**, **Kecepatan FPS**, **Keringanan Ukuran**, **Efisiensi Parameter**) berdampingan dengan grafik **Peringkat Skor Komposit Keseluruhan (0–100)**.

In [ ]:
ordered_df = summary_df.set_index("Metode").loc[METHOD_NAMES].reset_index()

max_fps = max(float(ordered_df["FPS"].max()), 1e-6)
min_size = min(float(ordered_df["Ukuran_MB"].min()), 1e9)
min_params = min(float(ordered_df["Parameter_Juta"].min()), 1e9)

radar_labels = (
    ["mAP@0.50", "F1-Score", "Kecepatan (FPS)", "Keringanan\nUkuran (MB)", "Efisiensi\nParameter"]
    if HAS_GROUND_TRUTH
    else ["Detection Rate", "Confidence", "Kecepatan (FPS)", "Keringanan\nUkuran (MB)", "Efisiensi\nParameter"]
)
angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()
angles_closed = angles + [angles[0]]

fig = plt.figure(figsize=(15, 5.8))
ax_radar = fig.add_subplot(1, 2, 1, polar=True)
ax_bar = fig.add_subplot(1, 2, 2)

for _, r in ordered_df.iterrows():
    m = r["Metode"]
    v1 = float(r["mAP50_pct"] if HAS_GROUND_TRUTH else r["Detection_Rate_pct"])
    v2 = float(r["F1_pct"] if HAS_GROUND_TRUTH else r["Mean_Confidence_pct"])
    v3 = (float(r["FPS"]) / max_fps) * 100.0
    v4 = (min_size / max(1e-6, float(r["Ukuran_MB"]))) * 100.0
    v5 = (min_params / max(1e-6, float(r["Parameter_Juta"]))) * 100.0
    vals = [v1, v2, v3, v4, v5]
    vals_closed = vals + [vals[0]]

    lw = 2.8 if m == METHOD_NAMES[0] else 1.9
    alpha_fill = 0.24 if m == METHOD_NAMES[0] else 0.10
    ax_radar.plot(angles_closed, vals_closed, color=METHOD_COLORS[m], linewidth=lw, marker="o", markersize=5, label=m)
    ax_radar.fill(angles_closed, vals_closed, color=METHOD_COLORS[m], alpha=alpha_fill)

ax_radar.set_xticks(angles)
ax_radar.set_xticklabels(radar_labels, fontsize=9.5, fontweight="bold")
ax_radar.set_ylim(0, 105)
ax_radar.set_yticks([25, 50, 75, 100])
ax_radar.set_yticklabels(["25", "50", "75", "100"], fontsize=8, color="#64748B")
ax_radar.set_title("Profil Multi-Kriteria (Skala 0–100)", pad=18)
ax_radar.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=1, frameon=True, fontsize=9)

ranked_df = summary_df.iloc[::-1].reset_index(drop=True)
y_labels = [f"{r['Peringkat']}  {METHOD_SHORT[r['Metode']].replace(chr(10), ' — ')}" for _, r in ranked_df.iterrows()]
bar_cols = [METHOD_COLORS[r["Metode"]] for _, r in ranked_df.iterrows()]
bars = ax_bar.barh(y_labels, ranked_df["Skor_Komposit"], color=bar_cols, height=0.52, edgecolor="#0F172A", linewidth=0.8)
ax_bar.set_xlim(0, 118)
ax_bar.set_xlabel("Skor Komposit Keseluruhan (0–100)")
ax_bar.set_title("Peringkat Akhir (50% Kualitas + 25% FPS + 25% Efisiensi)")
ax_bar.grid(axis="x", linestyle="--", alpha=0.35)
for bar, (_, r) in zip(bars, ranked_df.iterrows()):
    w_val = float(r["Skor_Komposit"])
    suffix = "  ★ PEMENANG" if r["Peringkat"] == "#1" else ""
    ax_bar.text(w_val + 1.5, bar.get_y() + bar.get_height() / 2, f"{w_val:.1f}{suffix}", va="center", fontweight="bold", fontsize=10)

plt.tight_layout()
save_fig(fig, "02_radar_dan_skor_komposit.png")
plt.show()

## Cell 8 — Perbandingan Visual Side-by-Side (`SSD MobileNet V2 FERQUEUE` vs `SSD MobileNet V1` vs `Faster R-CNN ResNet50`)

Menampilkan maksimal **4 gambar uji representatif** lintas kelas (`Mobil`, `Truk`, `Bus`) secara berdampingan beserta waktu inferensi (`ms`) masing-masing metode.

In [ ]:
def draw_detections_rgb(rgb: np.ndarray, dets: list[dict], show_score: bool = True) -> np.ndarray:
    canvas = rgb.copy()
    h, w = canvas.shape[:2]
    thick = max(2, int(round(min(h, w) / 180)))
    font_scale = max(0.55, min(h, w) / 650.0)
    for d in dets:
        x1, y1, x2, y2 = [int(round(v)) for v in d["xyxy_px"]]
        color = CLASS_COLORS.get(d["class"], (255, 255, 255))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thick)
        label = f"{d['class']} {d['score']*100:.0f}%" if show_score else d["class"]
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, max(1, thick - 1))
        ly = max(th + 8, y1)
        cv2.rectangle(canvas, (x1, ly - th - 8), (x1 + tw + 10, ly), color, -1)
        cv2.putText(
            canvas,
            label,
            (x1 + 5, ly - 4),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (15, 23, 42),
            max(1, thick - 1),
            cv2.LINE_AA,
        )
    return canvas


N_VISUAL = min(4, len(test_samples))
scored_indices = []
for idx, sample in enumerate(test_samples):
    fq_dets = predictions[METHOD_NAMES[0]][idx]
    p1_dets = predictions[METHOD_NAMES[1]][idx]
    p2_dets = predictions[METHOD_NAMES[2]][idx]
    fq_ok = 1 if len(fq_dets) > 0 else 0
    fq_conf = fq_dets[0]["score"] if fq_dets else 0.0
    fq_cls = fq_dets[0]["class"] if fq_dets else (sample["gt"][0]["class"] if sample["gt"] else "Lainnya")
    p1_conf = p1_dets[0]["score"] if p1_dets else 0.0
    p2_conf = p2_dets[0]["score"] if p2_dets else 0.0
    margin = fq_conf - 0.5 * (p1_conf + p2_conf)
    scored_indices.append((idx, fq_cls, fq_ok, margin, fq_conf))

scored_indices.sort(key=lambda t: (t[2], t[3], t[4]), reverse=True)
chosen_indices = []
used_classes = set()
for idx, cls_name, *_ in scored_indices:
    if cls_name not in used_classes:
        chosen_indices.append(idx)
        used_classes.add(cls_name)
    if len(chosen_indices) >= N_VISUAL:
        break
for idx, *_ in scored_indices:
    if idx not in chosen_indices:
        chosen_indices.append(idx)
    if len(chosen_indices) >= N_VISUAL:
        break

fig, axes = plt.subplots(len(chosen_indices), 3, figsize=(16.5, 4.4 * len(chosen_indices)))
if len(chosen_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_i, img_idx in enumerate(chosen_indices):
    sample = test_samples[img_idx]
    gt_txt = f" | GT: {sample['gt'][0]['class']}" if sample["gt"] else ""
    for col_i, m_name in enumerate(METHOD_NAMES):
        dets = predictions[m_name][img_idx]
        ms = latencies_ms[m_name][img_idx]
        vis = draw_detections_rgb(sample["rgb"], dets, show_score=True)
        det_summary = ", ".join(f"{d['class']}({d['score']*100:.0f}%)" for d in dets[:2]) or "Tidak Terdeteksi"
        short_name = METHOD_SHORT[m_name].replace("\n", " ")

        ax = axes[row_i, col_i]
        ax.imshow(vis)
        ax.axis("off")
        title_color = "#047857" if col_i == 0 else "#1E293B"
        ax.set_title(
            f"{short_name}\n{det_summary} | {ms:.1f} ms" + (gt_txt if col_i == 0 else ""),
            fontsize=9.5,
            fontweight="bold",
            color=title_color,
        )

plt.suptitle(
    "Perbandingan Deteksi Side-by-Side: SSD MobileNet V2 (FERQUEUE) vs SSD MobileNet V1 vs Faster R-CNN ResNet50",
    fontsize=13,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.975])
save_fig(fig, "03_perbandingan_visual_side_by_side.png")
plt.show()

## Cell 9 — Tabel Matriks Keputusan (300 DPI) & Unduh Paket ZIP Presentasi

Merender tabel ringkasan akhir (`04_matriks_keputusan_3_metode.png`), menyimpan ringkasan JSON, dan mengemas seluruh grafik + CSV ke 1 file ZIP (`nb6_mv2_vs_mv1_vs_resnet50.zip`) yang langsung terunduh di Google Colab.

In [ ]:
ordered_df = summary_df.copy()

display_cols = ["Peringkat", "Metode", "Backbone", "Tipe"]
if HAS_GROUND_TRUTH:
    display_cols += ["mAP50_pct", "F1_pct", "Mean_IoU_pct"]
else:
    display_cols += ["Detection_Rate_pct", "Mean_Confidence_pct"]
display_cols += ["Waktu_ms", "FPS", "Ukuran_MB", "Parameter_Juta", "Skor_Komposit"]

col_rename = {
    "Peringkat": "Rank",
    "Metode": "Metode Deteksi",
    "Backbone": "Backbone",
    "Tipe": "Arsitektur",
    "mAP50_pct": "mAP@0.50 (%)",
    "F1_pct": "F1 (%)",
    "Mean_IoU_pct": "IoU (%)",
    "Detection_Rate_pct": "Det. Rate (%)",
    "Mean_Confidence_pct": "Conf. (%)",
    "Waktu_ms": "Waktu (ms)",
    "FPS": "FPS",
    "Ukuran_MB": "Bobot (MB)",
    "Parameter_Juta": "Params (M)",
    "Skor_Komposit": "Skor Akhir",
}
tbl_df = ordered_df[display_cols].rename(columns=col_rename)

fig, ax = plt.subplots(figsize=(16.2, 3.5))
ax.axis("off")
ax.set_title(
    f"Matriks Perbandingan Akhir: SSD MobileNet V2 (FERQUEUE) vs SSD MobileNet V1 vs Faster R-CNN ResNet50 ({len(test_samples)} Gambar)",
    fontsize=12.5,
    fontweight="bold",
    pad=12,
)

table = ax.table(
    cellText=tbl_df.values,
    colLabels=tbl_df.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9.2)
table.scale(1.0, 2.0)

for (r_i, c_i), cell in table.get_celld().items():
    cell.set_edgecolor("#CBD5E1")
    if r_i == 0:
        cell.set_facecolor("#0C2744")
        cell.set_text_props(color="white", weight="bold")
    elif r_i == 1:
        cell.set_facecolor("#ECFDF5")
        cell.set_text_props(color="#065F46", weight="bold")
    else:
        cell.set_facecolor("#FFFFFF" if r_i % 2 == 0 else "#F8FAFC")
        cell.set_text_props(color="#0F172A")

plt.tight_layout()
save_fig(fig, "04_matriks_keputusan_3_metode.png")
plt.show()

summary_json = OUT_DIR / "05_ringkasan_mv2_vs_mv1_vs_resnet50.json"
summary_json.write_text(
    json.dumps(
        {
            "total_test_images": len(test_samples),
            "has_ground_truth": HAS_GROUND_TRUTH,
            "device": str(TORCH_DEVICE),
            "conf_threshold": CONF_TH,
            "nms_iou_threshold": NMS_IOU_TH,
            "backbone_benchmark": backbone_df.drop(columns=["Label_Plot", "Color"]).to_dict(orient="records"),
            "ranking": summary_df.to_dict(orient="records"),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

zip_path = OUT_DIR.parent / "nb6_mv2_vs_mv1_vs_resnet50.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("*")):
        if f.is_file():
            zf.write(f, arcname=f.name)

print(f"\nPaket ZIP siap: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
if IN_COLAB:
    colab_files.download(str(zip_path))